In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q \
  "transformers>=4.57.0" \
  accelerate \
  pillow \
  sentencepiece

In [3]:
from transformers import pipeline
from PIL import Image
pipe = pipeline(
    "image-text-to-text",
    model="Qwen/Qwen3-VL-2B-Instruct",
    device_map="auto",
    trust_remote_code=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [4]:
from google.colab import files
from PIL import Image

uploaded = files.upload()
path = next(iter(uploaded))
image = Image.open(path).convert("RGB")

Saving 000001.jpg to 000001.jpg


In [13]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {
                "type": "text",
                "text": ("Il s’agit d’une illustration médiévale. Analyse attentivement l’image et identifie uniquement les éléments visuels pertinents pour une annotation iconographique de type TIMEL. Concentre-toi sur des concepts visuels discrets et observables. Évite toute interprétation historique, narrative ou contextuelle non directement visible dans l’image. N’invente pas d’informations absentes de l’illustration. Indiquez uniquement label de timel"
                )
            }
        ]
    }
]

In [14]:
output = pipe(
    messages,
    max_new_tokens=256,
    num_beams=4
)

output

[{'input_text': [{'role': 'user',
    'content': [{'type': 'image',
      'image': <PIL.Image.Image image mode=RGB size=887x1351>},
     {'type': 'text',
      'text': 'Il s’agit d’une illustration médiévale. Analyse attentivement l’image et identifie uniquement les éléments visuels pertinents pour une annotation iconographique de type TIMEL. Concentre-toi sur des concepts visuels discrets et observables. Évite toute interprétation historique, narrative ou contextuelle non directement visible dans l’image. N’invente pas d’informations absentes de l’illustration. Indiquez uniquement label de timel'}]}],
  'generated_text': [{'role': 'user',
    'content': [{'type': 'image',
      'image': <PIL.Image.Image image mode=RGB size=887x1351>},
     {'type': 'text',
      'text': 'Il s’agit d’une illustration médiévale. Analyse attentivement l’image et identifie uniquement les éléments visuels pertinents pour une annotation iconographique de type TIMEL. Concentre-toi sur des concepts visuels di

In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq
from PIL import Image
import torch

model_name = "davanstrien/iconclass-vlm"

processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForVision2Seq.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
).eval()

preprocessor_config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.51G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},  # 关键：必须有这个占位符
            {"type": "text", "text": "Generate Iconclass labels for this image"}
        ],
    }
]

prompt = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = processor(
    text=prompt,
    images=[image],          # 关键：用 list
    return_tensors="pt"
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        num_beams=4
    )

print(processor.decode(outputs[0], skip_special_tokens=True))

system
You are a helpful assistant.
user
Generate Iconclass labels for this image
assistant
{"iconclass-codes": ["11D12", "11D121", "11D122", "11D123", "11D124", "11D125", "11D126", "11D127", "11D128", "11D129", "11D13", "11D14", "11D15", "11D16", "11D17", "11D18", "11D19", "11D192", "11D193", "11D194", "11D195", "11D196", "11D197", "11D198", "11D199", "11D2", "11D21", "11D22", "11D23", "11D24", "11D25", "11D26", "11D27", "11
